# Salon Extra Service Prediction

**Educational Machine Learning Project**

This project predicts whether a salon customer will buy an **extra service**.

### Project Goal
Build a simple classification model that takes customer information and returns the probability of buying an additional service.

### Features Used
| Feature | Description |
|---------|-------------|
| Age | Customer age |
| Visits | Number of previous visits |
| Bill_Amount | Current bill amount |
| Is_New_Customer | 1 = New customer, 0 = Returning |
| Service_Type | Haircut / Coloring / Treatment |
| Time_of_Day | Morning / Evening |

**Target:** `Bought_Extra_Service` (0 = No, 1 = Yes)

---

### Short Explanation (for presentation)

1. We created a synthetic dataset of 400 customers.
2. We split the data into training and testing sets.
3. We used a preprocessing pipeline (One-Hot Encoding for categorical features).
4. We trained two models: **Logistic Regression** and **Random Forest**.
5. We evaluated them using Accuracy, F1-Score, and ROC-AUC.
6. The best model is used to predict the probability of buying an extra service.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Create Dataset

We generate a synthetic dataset (400 customers) for educational purposes.

In [ ]:
np.random.seed(42)
n = 400

age = np.random.randint(18, 65, n)
visits = np.random.randint(1, 20, n)
bill_amount = np.round(np.random.uniform(50, 400, n), 1)
is_new_customer = np.random.choice([0, 1], n, p=[0.7, 0.3])
service_type = np.random.choice(["Haircut", "Coloring", "Treatment"], n, p=[0.45, 0.30, 0.25])
time_of_day = np.random.choice(["Morning", "Evening"], n, p=[0.55, 0.45])

# Create realistic probability for the target
prob = (
    0.15
    + 0.025 * visits
    + 0.0008 * bill_amount
    - 0.18 * is_new_customer
    + np.where(service_type == "Coloring", 0.12, 0)
    + np.where(service_type == "Treatment", 0.10, 0)
    + np.where(time_of_day == "Evening", 0.08, 0)
    - 0.002 * (age - 30)
)
prob = np.clip(prob, 0.05, 0.92)
bought_extra = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    "Age": age,
    "Visits": visits,
    "Bill_Amount": bill_amount,
    "Is_New_Customer": is_new_customer,
    "Service_Type": service_type,
    "Time_of_Day": time_of_day,
    "Bought_Extra_Service": bought_extra
})

print("Dataset created!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nTarget distribution:")
print(df["Bought_Extra_Service"].value_counts())

## 3. Features and Target

In [ ]:
TARGET = "Bought_Extra_Service"
X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = ["Age", "Visits", "Bill_Amount", "Is_New_Customer"]
categorical_features = ["Service_Type", "Time_of_Day"]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

## 5. Preprocessing Pipeline + Train Models

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        "model": pipe,
        "accuracy": acc,
        "f1": f1,
        "auc": auc
    }
    
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")
    print(f"Accuracy : {acc:.3f}")
    print(f"F1 Score : {f1:.3f}")
    print(f"ROC-AUC  : {auc:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

## 6. Select Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_name]["model"]

print("Best Model:", best_name)
print(f"ROC-AUC  : {results[best_name]['auc']:.3f}")
print(f"Accuracy : {results[best_name]['accuracy']:.3f}")
print(f"F1 Score : {results[best_name]['f1']:.3f}")

## 7. Simple Prediction Example

You can change the values below and re-run the cell to test different customers.

In [ ]:
# === Change these values to test ===
customer = {
    "Age": 32,
    "Visits": 8,
    "Bill_Amount": 180.0,
    "Is_New_Customer": 0,          # 0 = Returning, 1 = New
    "Service_Type": "Coloring",    # Haircut / Coloring / Treatment
    "Time_of_Day": "Evening"       # Morning / Evening
}

input_df = pd.DataFrame([customer])
probability = best_model.predict_proba(input_df)[0][1]
prediction = "Yes" if probability >= 0.5 else "No"

print("Customer Information:")
for k, v in customer.items():
    print(f"  {k}: {v}")

print(f"\nProbability of buying extra service: {probability*100:.1f}%")
print(f"Prediction: {prediction}")

## 8. Summary

- We built a complete classification pipeline.
- We compared Logistic Regression and Random Forest.
- The best model is used to predict the probability of buying an extra service.
- This project is educational and uses synthetic data.